# Averaged (across problems ≥ cutoff) good-reversal choice-probability plots — **fixed cumulative axis**

This notebook reproduces your script’s plot variants, **averaged across problems ≥ `CUTOFF_PROBLEM`**, and fixes two issues:

1. **Cumulative axis fraction removed is correct**  
   We compute `fraction_removed` **within each problem** (never by merging windows across problems), then average the fraction-removed curves across problems.

2. **Titles show real counts**  
   For each averaged plot:
   - `n_subjects` = union of subjects that contributed ≥1 good reversal in the included problems (or split)
   - `n_reversals` = total number of good reversals included (sum across problems)

We still use your existing `plot_choice_probs_around_good_reversals` for plots **without** cumulative fraction removed.
For plots **with** the fraction-removed axis, we use a small wrapper that draws the same lines + a twin axis.

In [1]:
# --- Imports / path setup (match your script) ---
import sys
from pathlib import Path
import numpy as np

sys.path.insert(0, str(Path.cwd().parent))

from src.behavior_import.import_data import *
from src.behavior_import.extract_trials import *
from src.behavior_analysis.get_total_reversals import *
from src.behavior_analysis.get_good_reversal_info import *
from src.behavior_analysis.get_choice_probs_around_good_reversals import *
from src.behavior_analysis.split_good_reversals_by_best_change import *
from src.behavior_analysis.split_early_late_good_reversals import *
from src.behavior_visualization.plot_choice_probs_around_good_reversals import *
from fix_grid_maze_cohort_02_problems import *


## Parameters (mirror your script)

In [2]:
task = "grid-maze"
# task = "open-field"

# which plot families to generate
DO_ALL = True
DO_SKIP = True
DO_MOVING_AVG = True
DO_REMOVE_BAD = True
DO_SPLIT_BY_BEST_CHANGE = False
DO_SPLIT_BY_BEST_CHANGE_AND_HALF = False
DO_SPLIT_BY_FIRST_TWO = True
DO_SPLIT_BY_HALF = True

# averaging across problems
CUTOFF_PROBLEM = 7   # average problems >= this

# window params
pre = 10
post = 40
skip_n_trials_after_reversal = 15

# smoothing
moving_avg_window = 4

# output base
OUT_BASE = Path("../results/figures")  # same as your script


## Load data and group into problems

In [3]:
folder_name = None
cohort = None
if task == "grid-maze":
    cohort = "cohort-02"
    folder_name = "3x3_maze_blocked_reward_bandit"
elif task == "open-field":
    cohort = "cohort-01"
    folder_name = "3x3_field_blocked_reward_bandit"

root = f"/Volumes/behrens/meg/{folder_name}/{cohort}/rawdata/"

subjects_data = import_data(root)
subjects_trials_by_problem = extract_trials_grouped_by_problem(subjects_data)

if task == "grid-maze" and cohort == "cohort-02":
    subjects_trials_by_problem = fix_grid_maze_cohort_02_problems(subjects_trials_by_problem)

problem_numbers = sorted(subjects_trials_by_problem.keys())
print("Problems found:", problem_numbers)
print("Averaging problems >=", CUTOFF_PROBLEM, ":", [p for p in problem_numbers if p >= CUTOFF_PROBLEM])


[WARN] Failed to read /Volumes/behrens/meg/3x3_maze_blocked_reward_bandit/cohort-02/rawdata/sub-03_id-MY_04_R/ses-27_date-20260124/behav/._MY_04_R-2026-01-24-124422.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /Volumes/behrens/meg/3x3_maze_blocked_reward_bandit/cohort-02/rawdata/sub-03_id-MY_04_R/ses-55_date-20260209/behav/._MY_04_R-2026-02-09-103918.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /Volumes/behrens/meg/3x3_maze_blocked_reward_bandit/cohort-02/rawdata/sub-03_id-MY_04_R/ses-28_date-20260124/behav/._MY_04_R-2026-01-24-165301.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /Volumes/behrens/meg/3x3_maze_blocked_reward_bandit/cohort-02/rawdata/sub-03_id-MY_04_R/ses-12_date-20260116/behav/._MY_04_R-2026-01-16-143640.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /Volum

## Helpers: per-problem curves + correct fraction-removed averaging

In [4]:
import matplotlib.pyplot as plt

KEYS = ("prev_best", "next_best", "third")

def compute_curve_from_windows(reversal_windows, *, pre, post, skip_n=0, do_moving_avg=False, moving_avg_window=4):
    """Returns (x, per_subject, across)."""
    x, per_subject, across = get_choice_probs_around_good_reversals(
        reversal_windows,
        pre=pre,
        post=post,
        skip_n_trials_after_reversal=skip_n
    )
    if do_moving_avg:
        x, per_subject_ma, across_ma = apply_moving_average_to_choice_probs(
            x, per_subject, moving_avg_window=moving_avg_window, mode="centered"
        )
        return x, per_subject_ma, across_ma
    return x, per_subject, across

def curves_by_problem_from_windows_dict(windows_dict, *, pre, post, skip_n=0, do_moving_avg=False, moving_avg_window=4):
    """Returns dict[p] = {x, across, subjects_set, num_reversals}."""
    out = {}
    for p, rw in windows_dict.items():
        if not rw:
            continue
        x, per_subject, across = compute_curve_from_windows(
            rw, pre=pre, post=post, skip_n=skip_n, do_moving_avg=do_moving_avg, moving_avg_window=moving_avg_window
        )
        if across.get("num_subjects", 0) == 0:
            continue
        out[p] = {
            "x": np.asarray(x, float),
            "across": across,
            "subjects": set(per_subject.keys()),
            "num_reversals": int(across.get("num_reversals", 0) or 0),
        }
    return out

def average_across_problem_curves(curves_by_problem):
    """Average curves treating each problem as one replicate; compute correct title counts."""
    probs = sorted(curves_by_problem.keys())
    if not probs:
        raise ValueError("No problems to average.")

    x0 = curves_by_problem[probs[0]]["x"]
    for p in probs[1:]:
        xp = curves_by_problem[p]["x"]
        if x0.shape != xp.shape or not np.allclose(x0, xp, equal_nan=True):
            raise ValueError(f"x mismatch between problems {probs[0]} and {p}. Ensure pre/post/skip match.")

    mean_out = {}
    se_out = {}
    for k in KEYS:
        mats = []
        for p in probs:
            y = curves_by_problem[p]["across"]["mean"].get(k)
            if y is None:
                continue
            mats.append(np.asarray(y, dtype=float))
        if not mats:
            mean_out[k] = None
            se_out[k] = None
            continue
        Y = np.vstack(mats)  # (n_problems, T)
        mean_out[k] = np.nanmean(Y, axis=0)
        if Y.shape[0] > 1:
            se_out[k] = np.nanstd(Y, axis=0, ddof=1) / np.sqrt(Y.shape[0])
        else:
            se_out[k] = np.zeros_like(mean_out[k])

    subjects_union = set()
    total_reversals = 0
    for p in probs:
        subjects_union |= curves_by_problem[p]["subjects"]
        total_reversals += int(curves_by_problem[p]["num_reversals"] or 0)

    across_avg = {
        "mean": mean_out,
        "se": se_out,
        "num_subjects": len(subjects_union),
        "num_reversals": total_reversals,
        "num_problems": len(probs),
        "problems_used": probs,
    }

    s = across_avg["mean"]["prev_best"] + across_avg["mean"]["next_best"] + across_avg["mean"]["third"]
    finite = np.isfinite(s)
    if finite.any():
        assert np.isclose(s[finite], 1.0, atol=1e-6).all(), "Averaged probabilities do not sum to 1."

    return x0, across_avg

def fraction_removed_by_problem(windows_dict, curves_by_problem, x, *, exclude_anchor=True):
    """Compute fraction_removed per problem (TOTAL mode) using *that problem's* windows, then average."""
    fracs = []
    probs_used = []
    for p in sorted(windows_dict.keys()):
        if p not in curves_by_problem:
            continue
        rw = windows_dict[p]
        if not rw:
            continue
        across_p = curves_by_problem[p]["across"]
        # function comes from your plotting module import: cumulative_total_events_over_post
        _, _, frac = cumulative_total_events_over_post(rw, x, across_p, exclude_anchor=exclude_anchor)
        fracs.append(np.asarray(frac, float))
        probs_used.append(p)

    if not fracs:
        return np.full_like(x, np.nan, dtype=float), probs_used

    F = np.vstack(fracs)
    frac_avg = np.nanmean(F, axis=0)
    return frac_avg, probs_used

def plot_choice_probs_with_fraction_removed_axis(x, across, frac_removed, *, save_path=None, title_extra=""):
    """Plot same main lines as plot_choice_probs_around_good_reversals + fraction removed on twin axis."""
    x = np.asarray(x, float)
    frac_removed = np.asarray(frac_removed, float)

    fig, ax = plt.subplots(figsize=(10, 6))

    # main lines
    for key, label in [("prev_best","Previous Best"),("next_best","New Best"),("third","Third Arm")]:
        y = np.asarray(across["mean"][key], float)
        s = np.asarray(across["se"][key], float)
        m = np.isfinite(x) & np.isfinite(y) & np.isfinite(s)
        if np.any(m):
            ax.plot(x[m], y[m], linewidth=2, color=COLOR_MAP[key], label=label)
            ax.fill_between(x, y - s, y + s, where=m, alpha=0.2, color=COLOR_MAP[key])

    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.axhline(1/3, color="gray", linestyle=":", linewidth=1)
    ax.set_ylim(0, 1)
    ax.set_xlim(x[0], x[-1])
    ax.set_xlabel("Trials from Good Reversal")
    ax.set_ylabel("Choice Probability")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # fraction removed axis
    ax2 = ax.twinx()
    m2 = np.isfinite(x) & np.isfinite(frac_removed)
    if np.any(m2):
        ax2.step(x[m2], frac_removed[m2], where="post", linewidth=2.0, color="#FF9100", alpha=0.7, label="Total Rev")
    ax2.set_ylabel("Fraction of Data Removed", rotation=-90, labelpad=15)
    ax2.set_ylim(0, 1)
    ax2.spines["top"].set_visible(False)

    title = (
        "Reversal-Aligned Choices\n"
        f"(mean ± se across problems | "
        f"n={across.get('num_subjects', 0)} subjects, "
        f"n={across.get('num_reversals', 0)} reversals, "
        f"n={across.get('num_problems', 0)} problems)\n"
    )
    if title_extra:
        title += str(title_extra)

    ax.set_title(title, pad=20)

    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=10)

    plt.tight_layout()

    if save_path:
        base = Path(save_path)
        base.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(str(base) + ".pdf", bbox_inches="tight")
        fig.savefig(str(base) + ".png", dpi=300, bbox_inches="tight")
    else:
        plt.show()
    plt.close(fig)


## Build per-problem reversal windows (and splits) for problems ≥ cutoff

In [5]:
windows_all = {}
windows_early_first2 = {}
windows_late_first2 = {}
windows_firsthalf = {}
windows_secondhalf = {}

windows_best2 = {}
windows_best3 = {}

for p in sorted(subjects_trials_by_problem.keys()):
    if p < CUTOFF_PROBLEM:
        continue

    subjects_trials = subjects_trials_by_problem[p]
    rw = get_good_reversal_info(subjects_trials, pre=pre, post=post, include_first_block=False)
    windows_all[p] = rw

    if DO_SPLIT_BY_FIRST_TWO:
        early2, late2 = split_good_reversals_early_late(rw, first_n=2)
        windows_early_first2[p] = early2
        windows_late_first2[p] = late2

    if DO_SPLIT_BY_HALF:
        firsthalf, secondhalf = split_good_reversals_early_late(rw)
        windows_firsthalf[p] = firsthalf
        windows_secondhalf[p] = secondhalf

    if DO_SPLIT_BY_BEST_CHANGE:
        b2, b3 = split_good_reversals_by_best_change(rw)
        windows_best2[p] = b2
        windows_best3[p] = b3

print("Built windows for problems:", sorted(windows_all.keys()))


Built windows for problems: [7, 8, 9, 10, 11]


## ALL averaged (with corrected fraction-removed cumulative axis)

In [6]:
if DO_ALL:
    curves_all = curves_by_problem_from_windows_dict(
        windows_all, pre=pre, post=post, skip_n=0, do_moving_avg=False, moving_avg_window=moving_avg_window
    )
    x_all_avg, across_all_avg = average_across_problem_curves(curves_all)

    frac_removed_all, probs_frac = fraction_removed_by_problem(windows_all, curves_all, x_all_avg, exclude_anchor=True)
    print("Fraction-removed computed from problems:", probs_frac)

    save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "Choice Probabilities Around Good Reversals (Averaged)"
    plot_choice_probs_with_fraction_removed_axis(
        x_all_avg, across_all_avg, frac_removed_all, save_path=save_path
    )


Fraction-removed computed from problems: [7, 8, 9, 10, 11]


/var/folders/g7/dhg2tq9173j2cwmlq6jvq28r0000gn/T/ipykernel_976/639469683.py:113: RuntimeWarning: Mean of empty slice
  frac_avg = np.nanmean(F, axis=0)


## ALL averaged — Early / Late (first 2) with corrected fraction-removed axis

In [7]:
if DO_ALL and DO_SPLIT_BY_FIRST_TWO and windows_early_first2:
    # Early
    curves_early = curves_by_problem_from_windows_dict(
        windows_early_first2, pre=pre, post=post, skip_n=0, do_moving_avg=False, moving_avg_window=moving_avg_window
    )
    x_early, across_early = average_across_problem_curves(curves_early)
    frac_removed_early, _ = fraction_removed_by_problem(windows_early_first2, curves_early, x_early, exclude_anchor=True)

    save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "Choice Probabilities Around Good Reversals (Averaged) (Early)"
    plot_choice_probs_with_fraction_removed_axis(x_early, across_early, frac_removed_early, save_path=save_path)

    # Late
    curves_late = curves_by_problem_from_windows_dict(
        windows_late_first2, pre=pre, post=post, skip_n=0, do_moving_avg=False, moving_avg_window=moving_avg_window
    )
    x_late, across_late = average_across_problem_curves(curves_late)
    frac_removed_late, _ = fraction_removed_by_problem(windows_late_first2, curves_late, x_late, exclude_anchor=True)

    save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "Choice Probabilities Around Good Reversals (Averaged) (Late)"
    plot_choice_probs_with_fraction_removed_axis(x_late, across_late, frac_removed_late, save_path=save_path)


/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:70: RuntimeWarning: Mean of empty slice
  "prev_best_mean": np.nanmean(prev_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:71: RuntimeWarning: Mean of empty slice
  "next_best_mean": np.nanmean(next_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:72: RuntimeWarning: Mean of empty slice
  "third_mean": np.nanmean(third_mat, axis=0),
/var/folders/g7/dhg2tq9173j2cwmlq6jvq28r0000gn/T/ipykernel_976/639469683.py:113: RuntimeWarning: Mean of empty slice
  frac_avg = np.nanmean(F, axis=0)
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:70: RuntimeWarning: Mean of empty slice
  "prev_best_mean": np.nanmean(prev_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get

[INFO] Skipping subject MY_04_R with no good reversals.


## SKIP family (averaged) — unchanged (no cumulative axis in your script)

In [8]:
if DO_SKIP:
    curves_skip = curves_by_problem_from_windows_dict(
        windows_all, pre=pre, post=post, skip_n=skip_n_trials_after_reversal, do_moving_avg=False, moving_avg_window=moving_avg_window
    )
    x_skip, across_skip = average_across_problem_curves(curves_skip)

    save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "Choice Probabilities Around Good Reversals, Skipping Initial Trials (Averaged)"
    plot_choice_probs_around_good_reversals(
        x_skip, across_skip,
        add_cumulative_axis=False,
        skip_n_trials_after_reversal=skip_n_trials_after_reversal,
        save_path=save_path
    )

    if DO_SPLIT_BY_FIRST_TWO and windows_early_first2:
        curves_skip_early = curves_by_problem_from_windows_dict(
            windows_early_first2, pre=pre, post=post, skip_n=skip_n_trials_after_reversal, do_moving_avg=False, moving_avg_window=moving_avg_window
        )
        x_skip_early, across_skip_early = average_across_problem_curves(curves_skip_early)

        save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "Choice Probabilities Around Good Reversals, Skipping Initial Trials (Averaged) (Early)"
        plot_choice_probs_around_good_reversals(
            x_skip_early, across_skip_early,
            add_cumulative_axis=False,
            skip_n_trials_after_reversal=skip_n_trials_after_reversal,
            save_path=save_path
        )

        curves_skip_late = curves_by_problem_from_windows_dict(
            windows_late_first2, pre=pre, post=post, skip_n=skip_n_trials_after_reversal, do_moving_avg=False, moving_avg_window=moving_avg_window
        )
        x_skip_late, across_skip_late = average_across_problem_curves(curves_skip_late)

        save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "Choice Probabilities Around Good Reversals, Skipping Initial Trials (Averaged) (Late)"
        plot_choice_probs_around_good_reversals(
            x_skip_late, across_skip_late,
            add_cumulative_axis=False,
            skip_n_trials_after_reversal=skip_n_trials_after_reversal,
            save_path=save_path
        )


/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:70: RuntimeWarning: Mean of empty slice
  "prev_best_mean": np.nanmean(prev_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:71: RuntimeWarning: Mean of empty slice
  "next_best_mean": np.nanmean(next_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:72: RuntimeWarning: Mean of empty slice
  "third_mean": np.nanmean(third_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:70: RuntimeWarning: Mean of empty slice
  "prev_best_mean": np.nanmean(prev_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:71: RuntimeWarning: Mean of empty slice
  "next_best_mean": np.nanmean(next_mat, axis=0),
/Users/megyoung/magnitu

[INFO] Skipping subject MY_04_R with no good reversals.


## MOVING AVERAGE family (averaged) with corrected fraction-removed axis

In [9]:
if DO_MOVING_AVG:
    curves_ma = curves_by_problem_from_windows_dict(
        windows_all, pre=pre, post=post, skip_n=0, do_moving_avg=True, moving_avg_window=moving_avg_window
    )
    x_ma, across_ma = average_across_problem_curves(curves_ma)
    frac_removed_ma, _ = fraction_removed_by_problem(windows_all, curves_ma, x_ma, exclude_anchor=True)

    save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "Choice Probabilities Around Good Reversals (Moving Average) (Averaged)"
    plot_choice_probs_with_fraction_removed_axis(x_ma, across_ma, frac_removed_ma, save_path=save_path)

    if DO_SPLIT_BY_HALF and windows_firsthalf:
        curves_ma_first = curves_by_problem_from_windows_dict(
            windows_firsthalf, pre=pre, post=post, skip_n=0, do_moving_avg=True, moving_avg_window=moving_avg_window
        )
        x_ma_first, across_ma_first = average_across_problem_curves(curves_ma_first)
        frac_removed_first, _ = fraction_removed_by_problem(windows_firsthalf, curves_ma_first, x_ma_first, exclude_anchor=True)

        save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "Choice Probabilities Around Good Reversals (Moving Average) (Averaged) (First Half)"
        plot_choice_probs_with_fraction_removed_axis(x_ma_first, across_ma_first, frac_removed_first, save_path=save_path)

        curves_ma_second = curves_by_problem_from_windows_dict(
            windows_secondhalf, pre=pre, post=post, skip_n=0, do_moving_avg=True, moving_avg_window=moving_avg_window
        )
        x_ma_second, across_ma_second = average_across_problem_curves(curves_ma_second)
        frac_removed_second, _ = fraction_removed_by_problem(windows_secondhalf, curves_ma_second, x_ma_second, exclude_anchor=True)

        save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "Choice Probabilities Around Good Reversals (Moving Average) (Averaged) (Second Half)"
        plot_choice_probs_with_fraction_removed_axis(x_ma_second, across_ma_second, frac_removed_second, save_path=save_path)

    if DO_SPLIT_BY_FIRST_TWO and windows_early_first2:
        curves_ma_early = curves_by_problem_from_windows_dict(
            windows_early_first2, pre=pre, post=post, skip_n=0, do_moving_avg=True, moving_avg_window=moving_avg_window
        )
        x_ma_early, across_ma_early = average_across_problem_curves(curves_ma_early)
        frac_removed_ma_early, _ = fraction_removed_by_problem(windows_early_first2, curves_ma_early, x_ma_early, exclude_anchor=True)

        save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "Choice Probabilities Around Good Reversals (Moving Average) (Averaged) (Early)"
        plot_choice_probs_with_fraction_removed_axis(x_ma_early, across_ma_early, frac_removed_ma_early, save_path=save_path)

        curves_ma_late = curves_by_problem_from_windows_dict(
            windows_late_first2, pre=pre, post=post, skip_n=0, do_moving_avg=True, moving_avg_window=moving_avg_window
        )
        x_ma_late, across_ma_late = average_across_problem_curves(curves_ma_late)
        frac_removed_ma_late, _ = fraction_removed_by_problem(windows_late_first2, curves_ma_late, x_ma_late, exclude_anchor=True)

        save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "Choice Probabilities Around Good Reversals (Moving Average) (Averaged) (Late)"
        plot_choice_probs_with_fraction_removed_axis(x_ma_late, across_ma_late, frac_removed_ma_late, save_path=save_path)


/var/folders/g7/dhg2tq9173j2cwmlq6jvq28r0000gn/T/ipykernel_976/639469683.py:113: RuntimeWarning: Mean of empty slice
  frac_avg = np.nanmean(F, axis=0)


[INFO] Skipping subject MY_04_R with no good reversals.


/var/folders/g7/dhg2tq9173j2cwmlq6jvq28r0000gn/T/ipykernel_976/639469683.py:113: RuntimeWarning: Mean of empty slice
  frac_avg = np.nanmean(F, axis=0)
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:70: RuntimeWarning: Mean of empty slice
  "prev_best_mean": np.nanmean(prev_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:71: RuntimeWarning: Mean of empty slice
  "next_best_mean": np.nanmean(next_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:72: RuntimeWarning: Mean of empty slice
  "third_mean": np.nanmean(third_mat, axis=0),
/Users/megyoung/magnitude-bandit-analysis/src/behavior_analysis/get_choice_probs_around_good_reversals.py:90: RuntimeWarning: Mean of empty slice
  across_mean[k] = np.nanmean(stack, axis=0)
/Users/megyoung/anaconda3/envs/magnitude-bandit-analysis/lib/python3.13/

[INFO] Skipping subject MY_04_R with no good reversals.


## Optional: BEST-CHANGE (moving average) with corrected fraction-removed axis

In [10]:
if DO_SPLIT_BY_BEST_CHANGE and DO_MOVING_AVG:
    # best -> second
    curves_b2 = curves_by_problem_from_windows_dict(
        windows_best2, pre=pre, post=post, skip_n=0, do_moving_avg=True, moving_avg_window=moving_avg_window
    )
    x_b2, across_b2 = average_across_problem_curves(curves_b2)
    frac_removed_b2, _ = fraction_removed_by_problem(windows_best2, curves_b2, x_b2, exclude_anchor=True)

    save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "Choice Probs Good Reversals (Best->Second) (Moving Average) (Averaged)"
    plot_choice_probs_with_fraction_removed_axis(x_b2, across_b2, frac_removed_b2, save_path=save_path)

    # best -> third
    curves_b3 = curves_by_problem_from_windows_dict(
        windows_best3, pre=pre, post=post, skip_n=0, do_moving_avg=True, moving_avg_window=moving_avg_window
    )
    x_b3, across_b3 = average_across_problem_curves(curves_b3)
    frac_removed_b3, _ = fraction_removed_by_problem(windows_best3, curves_b3, x_b3, exclude_anchor=True)

    save_path = OUT_BASE / task / cohort / f"problems-geq-{CUTOFF_PROBLEM:02d}" / "reversal-stats" / "Choice Probs Good Reversals (Best->Third) (Moving Average) (Averaged)"
    plot_choice_probs_with_fraction_removed_axis(x_b3, across_b3, frac_removed_b3, save_path=save_path)
